In [0]:
# Table names for all layers
BRONZE_TABLE = "workspace.default.bronze_rides"
SILVER_TABLE = "workspace.default.silver_rides"

print("Bronze table:", BRONZE_TABLE)
print("Silver table:", SILVER_TABLE)

In [0]:
# Read from our Bronze Delta table
df = spark.table(BRONZE_TABLE)

print(f"Rows loaded from Bronze: {df.count():,}")

In [0]:
from pyspark.sql.functions import col, unix_timestamp

# Calculate ride duration in minutes
df = df.withColumn(
    "ride_time_min",
    (unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60
)

# Count how many rows we're removing
invalid = df.filter((col("ride_time_min") <= 1) | (col("ride_time_min") >= 1440))
print(f"Rows with invalid duration: {invalid.count():,}")

# Keep only valid rides
df = df.filter((col("ride_time_min") > 1) & (col("ride_time_min") < 1440))
print(f"Rows remaining: {df.count():,}")

In [0]:
# Count rows with missing station info
missing_stations = df.filter(
    col("start_station_name").isNull() |
    col("end_station_name").isNull()   |
    col("start_station_id").isNull()   |
    col("end_station_id").isNull()
)
print(f"Rows with missing station info: {missing_stations.count():,}")

# Remove them
df = df.filter(
    col("start_station_name").isNotNull() &
    col("end_station_name").isNotNull()   &
    col("start_station_id").isNotNull()   &
    col("end_station_id").isNotNull()
)
print(f"Rows remaining: {df.count():,}")

In [0]:
# Remove rows where end coordinates are null or zero
missing_coords = df.filter(
    col("end_lat").isNull() |
    col("end_lng").isNull() |
    (col("end_lat") == 0)   |
    (col("end_lng") == 0)
)
print(f"Rows with missing coordinates: {missing_coords.count():,}")

# Remove them
df = df.filter(
    col("end_lat").isNotNull() &
    col("end_lng").isNotNull() &
    (col("end_lat") != 0)      &
    (col("end_lng") != 0)
)
print(f"Rows remaining: {df.count():,}")

In [0]:
from pyspark.sql.functions import (
    month, dayofweek, hour, date_format,
    when, round as spark_round
)

# Add all time-based feature columns
df = df.withColumn("hour_of_day", hour("started_at"))
df = df.withColumn("weekday", date_format("started_at", "EEE"))
df = df.withColumn("month_name", date_format("started_at", "MMM"))
df = df.withColumn("month_num", month("started_at"))
df = df.withColumn("ride_time_min", spark_round(col("ride_time_min"), 2))

# Add is_weekend flag
df = df.withColumn(
    "is_weekend",
    when(dayofweek("started_at").isin([1, 7]), True).otherwise(False)
)

# Add season
df = df.withColumn(
    "season",
    when(col("month_num").isin([12, 1, 2]), "Winter")
    .when(col("month_num").isin([3, 4, 5]), "Spring")
    .when(col("month_num").isin([6, 7, 8]), "Summer")
    .otherwise("Fall")
)

# Add is_holiday flag
holidays = [
    "2024-09-02", "2024-10-14", "2024-11-11", "2024-11-28",
    "2024-12-25", "2025-01-01", "2025-01-20", "2025-02-17",
    "2025-05-26", "2025-06-19", "2025-07-04", "2025-09-01"
]
from pyspark.sql.functions import to_date, lit
df = df.withColumn(
    "is_holiday",
    date_format("started_at", "yyyy-MM-dd").isin(holidays)
)

print("New columns added:")
new_cols = ["ride_time_min", "hour_of_day", "weekday",
            "month_name", "season", "is_weekend", "is_holiday"]
for c in new_cols:
    print(f"  - {c}")

display(df.select("started_at", *new_cols).limit(5))

In [0]:
# Verify all ride_ids are exactly 16 characters
from pyspark.sql.functions import length

ride_id_lengths = (df
    .groupBy(length("ride_id").alias("id_length"))
    .count()
    .orderBy("id_length")
)

print("Ride ID length distribution:")
display(ride_id_lengths)

In [0]:
from pyspark.sql.functions import lpad, when, concat, lit

# Add formatted hour column (e.g. "1 am", "2 pm") 
# alongside the numeric hour_of_day we already have
df = df.withColumn(
    "hour_label",
    when(col("hour_of_day") == 0, "12 am")
    .when(col("hour_of_day") < 12, 
        concat(col("hour_of_day").cast("string"), lit(" am")))
    .when(col("hour_of_day") == 12, "12 pm")
    .otherwise(
        concat((col("hour_of_day") - 12).cast("string"), lit(" pm")))
)

# Verify it looks right
print("Sample hour labels:")
display(
    df.select("started_at", "hour_of_day", "hour_label")
    .orderBy("hour_of_day")
    .limit(5)
)

In [0]:
# Save cleaned and enriched data as Silver Delta table
# Only write columns present in the current Silver table schema to avoid schema mismatch
silver_cols = [
    "ride_id", "rideable_type", "started_at", "ended_at", "start_station_name",
    "start_station_id", "end_station_name", "end_station_id", "start_lat", "start_lng", "end_lat", "end_lng",
    "member_casual", "ride_time_min", "hour_of_day", "weekday", "month_name",
    "month_num", "is_weekend", "season", "is_holiday", "hour_label"
]
(df
    .select(*silver_cols)
    .write
    .format("delta")
    .mode("overwrite")
    .option('mergeSchema', 'true').saveAsTable(SILVER_TABLE)
)

print(f"Silver table saved successfully!")
print(f"  Table: {SILVER_TABLE}")
print(f"  Rows:  {df.count():,}")
print(f"  Cols:  {len(silver_cols)}")

In [0]:
# Read back and verify
df_silver = spark.table(SILVER_TABLE)

print("Silver table summary:")
print(f"  Rows:    {df_silver.count():,}")
print(f"  Columns: {len(df_silver.columns)}")

print("\nColumn list:")
for c in df_silver.columns:
    print(f"  - {c}")

print("\nRider type breakdown:")
display(
    df_silver.groupBy("member_casual")
    .count()
    .orderBy("member_casual")
)